# INGESTION DOC WORD

## Docx2txtLoader

In [20]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("../data/word_files/Document_RAG_Word.docx")
docs = loader.load()


print("Nombre de documents :", len(docs))
print(docs[0].page_content[:500])
print(docs[0].metadata)

Nombre de documents : 1
Procédure interne – Ingestion de documents Word pour un pipeline RAG

Ce document est conçu spécialement pour accompagner la narration de la section dédiée à l’ingestion des fichiers Word (.docx) dans un pipeline RAG professionnel. Il contient des structures réalistes : titres, sous-titres, listes, tableaux, paragraphes longs et séparations logiques. L’objectif est de démontrer les limites et les forces des loaders Docx2txt et Unstructured dans un contexte réel.

1. Contexte général

Les entrepr
{'source': '../data/word_files/Document_RAG_Word.docx'}


## version avancée : Quand Docx2txt ne suffit pas 

In [21]:
from langchain_community.document_loaders import UnstructuredWordDocumentLoader

loader = UnstructuredWordDocumentLoader("../data/word_files/Document_RAG_Word.docx")
docs = loader.load()

print("Nombre de documents :", len(docs))
print(docs[0].page_content[:500])
print(docs[0].metadata)

Nombre de documents : 1
Procédure interne – Ingestion de documents Word pour un pipeline RAG

Ce document est conçu spécialement pour accompagner la narration de la section dédiée à l’ingestion des fichiers Word (.docx) dans un pipeline RAG professionnel. Il contient des structures réalistes : titres, sous-titres, listes, tableaux, paragraphes longs et séparations logiques. L’objectif est de démontrer les limites et les forces des loaders Docx2txt et Unstructured dans un contexte réel.

1. Contexte général

Les entrepr
{'source': '../data/word_files/Document_RAG_Word.docx'}


# Atelier pratique : Nettoyage intelligent

In [22]:
import re

def clean_word_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)      # espaces multiples
    text = re.sub(r"\n+", "\n", text)     # sauts de ligne multiples
    text = text.replace("•", "-")         # normalisation puces
    text = text.strip()
    return text


# Pipeline complet word + clean + Split 

## 3 étapes :

1. Charger le document word 
2. le nettoyer proprement
3. le découper en chunks intelligents, adaptés aux embeddings 


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def process_word_file(path):
        # 1) Charger le document Word
    docs = Docx2txtLoader(path).load()
        # 2) Nettoyer le texte extrait
    cleaned = []
    for d in docs:
        d.page_content = clean_word_text(d.page_content)
        cleaned.append(d)
        # 3) Découpage intelligent en chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = splitter.split_documents(cleaned)
    return chunks


### Application de la fonction !

In [26]:
chunks = process_word_file("../data/word_files/Document_RAG_Word.docx")

print("Nombre de chunks :", len(chunks))
print(chunks[0].page_content)
print(chunks[0].metadata)

Nombre de chunks : 3
Procédure interne – Ingestion de documents Word pour un pipeline RAG Ce document est conçu spécialement pour accompagner la narration de la section dédiée à l’ingestion des fichiers Word (.docx) dans un pipeline RAG professionnel. Il contient des structures réalistes : titres, sous-titres, listes, tableaux, paragraphes longs et séparations logiques. L’objectif est de démontrer les limites et les forces des loaders Docx2txt et Unstructured dans un contexte réel. 1. Contexte général Les entreprises produisent un volume important de documents Word au quotidien. Ils contiennent des informations essentielles pour un système RAG, comme des procédures internes, des notes de cadrage, des comptes rendus de réunion et des dossiers techniques. Ces documents doivent être ingérés, nettoyés, découpés en chunks et indexés pour permettre à un modèle de répondre de façon cohérente aux requêtes des utilisateurs. 2
{'source': '../data/word_files/Document_RAG_Word.docx'}


In [27]:
for i, c in enumerate(chunks[:3]):
    print("="*80)
    print(f"CHUNK {i+1}")
    print("="*80)
    print(c.page_content[:500], "\n")

CHUNK 1
Procédure interne – Ingestion de documents Word pour un pipeline RAG Ce document est conçu spécialement pour accompagner la narration de la section dédiée à l’ingestion des fichiers Word (.docx) dans un pipeline RAG professionnel. Il contient des structures réalistes : titres, sous-titres, listes, tableaux, paragraphes longs et séparations logiques. L’objectif est de démontrer les limites et les forces des loaders Docx2txt et Unstructured dans un contexte réel. 1. Contexte général Les entreprise 

CHUNK 2
. 2. Challenges rencontrés Voici les défis typiques rencontrés lors de l'extraction de texte Word : - Multiples sauts de ligne - Listes mal reconstruites - Tableaux transformés en texte - Titres non reconnus - Retours chariots inutiles 3. Exemple de contenu complexe Le paragraphe suivant illustre un contenu souvent rencontré dans les entreprises. Il contient des descriptions opérationnelles, des éléments techniques et des blocs narratifs importants pour tester le comportement 

## Création d'une classe pour la pipeline Word

In [28]:
class WordIngestion:
    def __init__(self, chunk_size = 1000, chunk_overlap = 150):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap,
            separators=["\n\n", "\n", ".", " ", ""]
        )

    def _clean(self, text : str) -> str:
        return clean_word_text(text)

    def process(self, path : str):
        docs = Docx2txtLoader(path).load()
        for d in docs:
            d.page_content = self._clean(d.page_content)
        return self.splitter.split_documents(docs)

# Application

In [29]:
ingestion_word = WordIngestion(
    chunk_size=900,
    chunk_overlap=120
)

In [31]:
chunks = ingestion_word.process("../data/word_files/Document_RAG_Word.docx")

In [32]:
len(chunks)

4

In [ ]:
chunks[0].page_content

'Procédure interne – Ingestion de documents Word pour un pipeline RAG Ce document est conçu spécialement pour accompagner la narration de la section dédiée à l’ingestion des fichiers Word (.docx) dans un pipeline RAG professionnel. Il contient des structures réalistes : titres, sous-titres, listes, tableaux, paragraphes longs et séparations logiques. L’objectif est de démontrer les limites et les forces des loaders Docx2txt et Unstructured dans un contexte réel. 1. Contexte général Les entreprises produisent un volume important de documents Word au quotidien. Ils contiennent des informations essentielles pour un système RAG, comme des procédures internes, des notes de cadrage, des comptes rendus de réunion et des dossiers techniques'

In [35]:
chunks[0].metadata

{'source': '../data/word_files/Document_RAG_Word.docx'}

In [36]:
for i, c in enumerate(chunks[:3]):
    print("="*80)
    print(f"CHUNK {i+1}")
    print("="*80)
    print(c.page_content[:500], "\n")

CHUNK 1
Procédure interne – Ingestion de documents Word pour un pipeline RAG Ce document est conçu spécialement pour accompagner la narration de la section dédiée à l’ingestion des fichiers Word (.docx) dans un pipeline RAG professionnel. Il contient des structures réalistes : titres, sous-titres, listes, tableaux, paragraphes longs et séparations logiques. L’objectif est de démontrer les limites et les forces des loaders Docx2txt et Unstructured dans un contexte réel. 1. Contexte général Les entreprise 

CHUNK 2
. Ces documents doivent être ingérés, nettoyés, découpés en chunks et indexés pour permettre à un modèle de répondre de façon cohérente aux requêtes des utilisateurs. 2. Challenges rencontrés Voici les défis typiques rencontrés lors de l'extraction de texte Word : - Multiples sauts de ligne - Listes mal reconstruites - Tableaux transformés en texte - Titres non reconnus - Retours chariots inutiles 3. Exemple de contenu complexe Le paragraphe suivant illustre un contenu souvent

# Conclusion